# Titanic Dataset Analysis & Modeling

This notebook continues from the preprocessing pipeline:
1. **Exploratory Data Analysis (EDA)**
2. **Feature Visualization & Insights**
3. **Predictive Modeling**
4. **Model Evaluation & Feature Importance**
5. **Dashboard Preparation**

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

---
## Step 1: Load & Prepare Data

In [ ]:
# Load the dataset
df = pd.read_csv('train.csv')

# Display basic info
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

---
## Step 2: Exploratory Data Analysis (EDA)

### 2.1 - Overall Survival Statistics

In [ ]:
# Survival summary
survival_counts = df['Survived'].value_counts()
survival_rate = df['Survived'].mean()

print("=" * 50)
print("SURVIVAL SUMMARY")
print("=" * 50)
print(f"Total Passengers: {len(df)}")
print(f"Survived: {survival_counts[1]} ({survival_counts[1]/len(df)*100:.2f}%)")
print(f"Did Not Survive: {survival_counts[0]} ({survival_counts[0]/len(df)*100:.2f}%)")
print(f"\nSurvival Rate: {survival_rate:.2%}")

### 2.2 - Survival by Gender

In [ ]:
# Survival by Gender
gender_survival = pd.crosstab(df['Sex'], df['Survived'], margins=True)
gender_survival_pct = pd.crosstab(df['Sex'], df['Survived'], normalize='index') * 100

print("\nSURVIVAL BY GENDER (Counts)")
print(gender_survival)
print("\nSURVIVAL BY GENDER (Percentages)")
print(gender_survival_pct.round(2))

### 2.3 - Survival by Passenger Class

In [ ]:
# Survival by Passenger Class
class_survival = pd.crosstab(df['Pclass'], df['Survived'], margins=True)
class_survival_pct = pd.crosstab(df['Pclass'], df['Survived'], normalize='index') * 100

print("\nSURVIVAL BY PASSENGER CLASS (Counts)")
print(class_survival)
print("\nSURVIVAL BY PASSENGER CLASS (Percentages)")
print(class_survival_pct.round(2))

### 2.4 - Age & Fare Statistics

In [ ]:
# Age statistics by survival
print("\nAGE STATISTICS")
age_stats = df.groupby('Survived')['Age'].describe()
print(age_stats)

# Fare statistics by survival
print("\n\nFARE STATISTICS")
fare_stats = df.groupby('Survived')['Fare'].describe()
print(fare_stats)

### 2.5 - Embarkation Port Analysis

In [ ]:
# Survival by Embarked port
port_survival = pd.crosstab(df['Embarked'], df['Survived'], margins=True)
port_survival_pct = pd.crosstab(df['Embarked'], df['Survived'], normalize='index') * 100

print("\nSURVIVAL BY EMBARKATION PORT (Counts)")
print(port_survival)
print("\nSURVIVAL BY EMBARKATION PORT (Percentages)")
print(port_survival_pct.round(2))

# Port mapping
port_names = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
print("\nPort Codes:")
for code, name in port_names.items():
    print(f"  {code}: {name}")

---
## Step 3: Data Preprocessing (from previous notebook)

In [ ]:
# Make a copy for processing
df_processed = df.copy()

# Handle missing values
df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)
df_processed.drop(columns=['Cabin'], inplace=True, errors='ignore')
df_processed['Embarked'].fillna(df_processed['Embarked'].mode()[0], inplace=True)

# Encode Sex
df_processed['Sex'] = df_processed['Sex'].map({'male': 0, 'female': 1})

# One-hot encode Embarked
df_processed = pd.get_dummies(df_processed, columns=['Embarked'], drop_first=False)

# Normalize Age and Fare
scaler = MinMaxScaler()
df_processed[['Age', 'Fare']] = scaler.fit_transform(df_processed[['Age', 'Fare']])

# Create feature matrix and target
drop_cols = ['Survived', 'Name', 'Ticket', 'PassengerId']
X = df_processed.drop(columns=drop_cols)
y = df_processed['Survived']

print("Feature columns:", X.columns.tolist())
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")

---
## Step 4: Train-Test Split

In [ ]:
# Split data: 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size  : {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"\nTraining set class distribution:\n{y_train.value_counts()}")
print(f"\nValidation set class distribution:\n{y_val.value_counts()}")

---
## Step 5: Feature Correlation Analysis

In [ ]:
# Correlation with target
correlation = X_train.copy()
correlation['Survived'] = y_train
corr_matrix = correlation.corr()

# Get top correlations with Survived
survived_corr = corr_matrix['Survived'].sort_values(ascending=False)
print("\nFeature Correlation with Survival:")
print(survived_corr)

# Visualize full correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nCorrelation matrix saved as 'correlation_matrix.png'")

---
## Step 6: Build & Train Models

### 6.1 - Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train)
lr_val_pred = lr_model.predict(X_val)
lr_val_proba = lr_model.predict_proba(X_val)[:, 1]

# Evaluate
print("LOGISTIC REGRESSION RESULTS")
print("="*50)
print(f"Training Accuracy  : {accuracy_score(y_train, lr_train_pred):.4f}")
print(f"Validation Accuracy: {accuracy_score(y_val, lr_val_pred):.4f}")
print(f"Precision          : {precision_score(y_val, lr_val_pred):.4f}")
print(f"Recall             : {recall_score(y_val, lr_val_pred):.4f}")
print(f"F1-Score           : {f1_score(y_val, lr_val_pred):.4f}")
print(f"ROC-AUC            : {roc_auc_score(y_val, lr_val_proba):.4f}")

### 6.2 - Random Forest Classifier

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]

# Evaluate
print("RANDOM FOREST RESULTS")
print("="*50)
print(f"Training Accuracy  : {accuracy_score(y_train, rf_train_pred):.4f}")
print(f"Validation Accuracy: {accuracy_score(y_val, rf_val_pred):.4f}")
print(f"Precision          : {precision_score(y_val, rf_val_pred):.4f}")
print(f"Recall             : {recall_score(y_val, rf_val_pred):.4f}")
print(f"F1-Score           : {f1_score(y_val, rf_val_pred):.4f}")
print(f"ROC-AUC            : {roc_auc_score(y_val, rf_val_proba):.4f}")

### 6.3 - Gradient Boosting Classifier

In [ ]:
# Train Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)

# Predictions
gb_train_pred = gb_model.predict(X_train)
gb_val_pred = gb_model.predict(X_val)
gb_val_proba = gb_model.predict_proba(X_val)[:, 1]

# Evaluate
print("GRADIENT BOOSTING RESULTS")
print("="*50)
print(f"Training Accuracy  : {accuracy_score(y_train, gb_train_pred):.4f}")
print(f"Validation Accuracy: {accuracy_score(y_val, gb_val_pred):.4f}")
print(f"Precision          : {precision_score(y_val, gb_val_pred):.4f}")
print(f"Recall             : {recall_score(y_val, gb_val_pred):.4f}")
print(f"F1-Score           : {f1_score(y_val, gb_val_pred):.4f}")
print(f"ROC-AUC            : {roc_auc_score(y_val, gb_val_proba):.4f}")

---
## Step 7: Model Comparison & Selection

In [ ]:
# Compile results
models_summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [
        accuracy_score(y_val, lr_val_pred),
        accuracy_score(y_val, rf_val_pred),
        accuracy_score(y_val, gb_val_pred)
    ],
    'Precision': [
        precision_score(y_val, lr_val_pred),
        precision_score(y_val, rf_val_pred),
        precision_score(y_val, gb_val_pred)
    ],
    'Recall': [
        recall_score(y_val, lr_val_pred),
        recall_score(y_val, rf_val_pred),
        recall_score(y_val, gb_val_pred)
    ],
    'F1-Score': [
        f1_score(y_val, lr_val_pred),
        f1_score(y_val, rf_val_pred),
        f1_score(y_val, gb_val_pred)
    ],
    'ROC-AUC': [
        roc_auc_score(y_val, lr_val_proba),
        roc_auc_score(y_val, rf_val_proba),
        roc_auc_score(y_val, gb_val_proba)
    ]
})

print("\nMODEL COMPARISON")
print("="*90)
print(models_summary.to_string(index=False))
print("\nBest Model by ROC-AUC:", models_summary.loc[models_summary['ROC-AUC'].idxmax(), 'Model'])

---
## Step 8: Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFEATURE IMPORTANCE (Random Forest)")
print("="*50)
print(feature_importance.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nFeature importance saved as 'feature_importance.png'")

---
## Step 9: Confusion Matrices & ROC Curves

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = [
    ('Logistic Regression', lr_val_pred),
    ('Random Forest', rf_val_pred),
    ('Gradient Boosting', gb_val_pred)
]

for idx, (name, pred) in enumerate(models):
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name}\nConfusion Matrix')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print("Confusion matrices saved as 'confusion_matrices.png'")

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 7))

models_roc = [
    ('Logistic Regression', lr_val_proba),
    ('Random Forest', rf_val_proba),
    ('Gradient Boosting', gb_val_proba)
]

for name, proba in models_roc:
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("ROC curves saved as 'roc_curves.png'")

---
## Step 10: Summary & Key Insights

In [ ]:
print("\n" + "="*70)
print("TITANIC ANALYSIS SUMMARY & KEY INSIGHTS")
print("="*70)

print("\n1. SURVIVAL DEMOGRAPHICS:")
print(f"   - Overall survival rate: {survival_rate:.2%}")
print(f"   - Gender bias: Women had ~74% survival rate, men ~19%")
print(f"   - Class impact: 1st class ~63%, 2nd class ~47%, 3rd class ~24%")

print("\n2. TOP FEATURES (by importance):")
for idx, row in feature_importance.head(5).iterrows():
    print(f"   - {row['Feature']}: {row['Importance']:.4f}")

print("\n3. BEST PERFORMING MODEL:")
best_idx = models_summary['ROC-AUC'].idxmax()
best_row = models_summary.iloc[best_idx]
print(f"   - Model: {best_row['Model']}")
print(f"   - Accuracy: {best_row['Accuracy']:.4f}")
print(f"   - ROC-AUC: {best_row['ROC-AUC']:.4f}")
print(f"   - F1-Score: {best_row['F1-Score']:.4f}")

print("\n4. RECOMMENDATIONS:")
print("   - Deploy the Random Forest or Gradient Boosting model")
print("   - Focus on Sex, Passenger Class, and Age as key features")
print("   - Monitor model performance on new data")
print("   - Consider ensemble methods for production")

print("\n" + "="*70)